# Smart-Truck Agent Classifier
**Task**: Train a deep learning model to classify user queries into 10 agent classes for routing in a LangGraph chatboard.

**Architecture**: Fine-tuned `all-MiniLM-L6-v2` (22M params) + auxiliary entity features + classification head.

**Dataset**: 15,000 balanced rows (1,500 per agent class).

**Runtime**: T4 GPU (Colab free tier) — ~10 min training.

In [ ]:
!pip install -q transformers sentence-transformers accelerate

In [ ]:
import torch
import numpy as np
import pandas as pd
import csv, io, re, json, os, random
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. Load & Clean Data

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload dataset_balanced.csv

In [ ]:
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Clean
df = df.dropna(subset=['query', 'agent'])
df = df[df['agent'] != 'agent']  # remove stray headers
df['query'] = df['query'].str.strip().str.strip('"')
df = df[df['query'].str.len() > 3]
df = df.drop_duplicates(subset='query', keep='first')
df = df.reset_index(drop=True)

print(f'Loaded {len(df)} rows')
print(f'\nClass distribution:')
print(df['agent'].value_counts().to_string())

## 2. EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['agent'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Samples per Agent Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

df['query'].str.len().hist(bins=50, ax=axes[1], color='coral')
axes[1].set_title('Query Length Distribution')
axes[1].set_xlabel('Characters')

plt.tight_layout()
plt.show()

print(f"Query length — min: {df['query'].str.len().min()}, "
      f"max: {df['query'].str.len().max()}, "
      f"mean: {df['query'].str.len().mean():.1f}")

## 3. Entity Feature Extraction
Regex-based auxiliary features that boost classification accuracy.

In [ ]:
VEHICLE_RE = re.compile(r'[A-Z]{2}\d{2}[A-Z]{1,2}\d{4}', re.IGNORECASE)
DRIVER_ID_RE = re.compile(r'\bdriver\s+\d+\b', re.IGNORECASE)
TRIP_ID_RE = re.compile(r'\btrip\s+\d+\b', re.IGNORECASE)
DATE_RE = re.compile(
    r'\d{4}-\d{2}-\d{2}'
    r'|\b(?:january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{4}\b'
    r'|\b(?:last|this|past|next)\s+(?:week|month|quarter|year)\b'
    r'|\b(?:today|yesterday|tomorrow|kal|aaj)\b'
    r'|\b(?:last|past)\s+\d+\s+days\b',
    re.IGNORECASE
)
ROUTE_RE = re.compile(r'\bfrom\s+\w+\s+to\s+\w+|\w+\s+to\s+\w+\s+route|\w+\s+se\s+\w+', re.IGNORECASE)


def extract_aux_features(text):
    t = text.lower()
    return [
        # Entity detectors
        1.0 if VEHICLE_RE.search(text) else 0.0,
        1.0 if DRIVER_ID_RE.search(text) else 0.0,
        1.0 if TRIP_ID_RE.search(text) else 0.0,
        1.0 if DATE_RE.search(text) else 0.0,
        1.0 if ROUTE_RE.search(text) else 0.0,
        # Intent keyword detectors
        1.0 if any(w in t for w in ['predict', 'forecast', 'estimate', 'expected', 'will it', 'probability', 'how long will']) else 0.0,
        1.0 if any(w in t for w in ['optimize', 'best route', 'recommend', 'suggest', 'optimal', 'alternative', 'assign']) else 0.0,
        1.0 if any(w in t for w in ['alert', 'anomaly', 'unusual', 'suspicious', 'outlier', 'warning', 'scan', 'flag']) else 0.0,
        1.0 if any(w in t for w in ['fatigue', 'tired', 'rest', 'safety', 'workload', 'consecutive days', 'hours driving', 'overwork']) else 0.0,
        1.0 if any(w in t for w in ['fleet', 'dashboard', 'overall', 'kpi', 'fleet-wide']) else 0.0,
        1.0 if any(w in t for w in ['demand', 'volume', 'growth', 'seasonal', 'client forecast', 'next week', 'client profile']) else 0.0,
        1.0 if any(w in t for w in ['vehicle', 'asset', 'truck']) else 0.0,
        # Normalized length features
        min(len(text) / 100.0, 1.0),
        min(len(text.split()) / 20.0, 1.0),
    ]


AUX_DIM = 14
aux_matrix = np.array([extract_aux_features(q) for q in df['query']], dtype=np.float32)

aux_names = ['vehicle_id', 'driver_id', 'trip_id', 'date', 'route_pair',
             'predict', 'optimize', 'alert', 'safety', 'fleet', 'demand',
             'vehicle_kw', 'norm_len', 'norm_words']

print('Auxiliary features per-column sum (signal strength):')
for name, total in zip(aux_names, aux_matrix.sum(axis=0)):
    print(f'  {name:<15s}: {total:>7.0f}')

## 4. Label Encoding & Train/Val/Test Split

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['agent'])
NUM_CLASSES = len(le.classes_)

print(f'{NUM_CLASSES} classes:')
for i, name in enumerate(le.classes_):
    print(f'  {i}: {name} ({(df["label"]==i).sum()} samples)')

# Stratified 80/10/10 split
train_idx, temp_idx = train_test_split(
    np.arange(len(df)), test_size=0.2,
    stratify=df['label'], random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5,
    stratify=df['label'].iloc[temp_idx], random_state=42
)

print(f'\nTrain: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}')

In [ ]:
# Class weights for imbalanced loss
class_weights = compute_class_weight(
    'balanced',
    classes=np.arange(NUM_CLASSES),
    y=df['label'].iloc[train_idx].values
)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(device)
print('Class weights:')
for i, (name, w) in enumerate(zip(le.classes_, class_weights)):
    print(f'  {name}: {w:.3f}')

## 5. Tokenization & PyTorch Dataset

In [ ]:
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
MAX_LEN = 64  # queries are short
BATCH_SIZE = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class AgentDataset(Dataset):
    def __init__(self, indices):
        self.queries = df['query'].iloc[indices].tolist()
        self.labels = df['label'].iloc[indices].tolist()
        self.aux = torch.tensor(aux_matrix[indices], dtype=torch.float32)

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.queries[idx],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'aux': self.aux[idx],
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }


train_loader = DataLoader(AgentDataset(train_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(AgentDataset(val_idx), batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
test_loader = DataLoader(AgentDataset(test_idx), batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

## 6. Model Architecture
`MiniLM encoder` → mean pooling → concat with `14 aux features` → MLP → `10 classes`

In [ ]:
class AgentClassifier(nn.Module):
    def __init__(self, model_name, num_classes, aux_dim, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size  # 384

        self.drop = nn.Dropout(dropout)
        combined = hidden + aux_dim  # 384 + 14 = 398

        self.head = nn.Sequential(
            nn.Linear(combined, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask, aux):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Mean pooling over non-padding tokens
        tokens = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (tokens * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        pooled = self.drop(pooled)
        combined = torch.cat([pooled, aux], dim=1)
        return self.head(combined)


model = AgentClassifier(MODEL_NAME, NUM_CLASSES, AUX_DIM).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable:    {trainable:,}')
print(f'Model size:   ~{total_params * 4 / 1e6:.1f} MB (FP32)')

## 7. Training

In [ ]:
EPOCHS = 15
LR = 2e-5
PATIENCE = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(weight=class_weights_t)
scaler = torch.amp.GradScaler('cuda')

total_steps = len(train_loader) * EPOCHS
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=total_steps, pct_start=0.1
)

best_val_acc = 0.0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0

    for batch in train_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        aux = batch['aux'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(ids, mask, aux)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        t_loss += loss.item() * labels.size(0)
        t_correct += (logits.argmax(1) == labels).sum().item()
        t_total += labels.size(0)

    # ── Validate ──
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0

    with torch.no_grad():
        for batch in val_loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            aux = batch['aux'].to(device)
            labels = batch['label'].to(device)

            with torch.amp.autocast('cuda'):
                logits = model(ids, mask, aux)
                loss = criterion(logits, labels)

            v_loss += loss.item() * labels.size(0)
            v_correct += (logits.argmax(1) == labels).sum().item()
            v_total += labels.size(0)

    t_acc = t_correct / t_total
    v_acc = v_correct / v_total
    history['train_loss'].append(t_loss / t_total)
    history['val_loss'].append(v_loss / v_total)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    marker = ''
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), 'best_model.pt')
        patience_counter = 0
        marker = ' << saved'
    else:
        patience_counter += 1

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
          f'Train {t_loss/t_total:.4f} / {t_acc:.4f} | '
          f'Val {v_loss/v_total:.4f} / {v_acc:.4f}{marker}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1}')
        break

print(f'\nBest validation accuracy: {best_val_acc:.4f}')

## 8. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train')
ax1.plot(epochs_range, history['val_loss'], 'r-o', label='Val')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], 'b-o', label='Train')
ax2.plot(epochs_range, history['val_acc'], 'r-o', label='Val')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Test Set Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('best_model.pt', weights_only=True))
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        aux = batch['aux'].to(device)
        labels = batch['label'].to(device)

        with torch.amp.autocast('cuda'):
            logits = model(ids, mask, aux)

        probs = torch.softmax(logits.float(), dim=1)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

print('=== CLASSIFICATION REPORT ===')
print(classification_report(all_labels, all_preds, target_names=le.classes_, digits=3))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Test Set')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Confidence Threshold Analysis
Determines routing thresholds for production.

In [ ]:
max_probs = all_probs.max(axis=1)
correct = all_preds == all_labels

print(f"{'Threshold':<12} {'Coverage':<12} {'Accuracy':<12} {'Routed':<12} {'Correct':<12}")
print('-' * 60)
for t in [0.3, 0.5, 0.7, 0.8, 0.85, 0.9, 0.95]:
    mask = max_probs >= t
    n = mask.sum()
    acc = correct[mask].mean() if n > 0 else 0
    print(f'{t:<12.2f} {mask.mean():<12.1%} {acc:<12.1%} {n:<12d} {correct[mask].sum():<12d}')

# Distribution of confidences for correct vs incorrect
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(max_probs[correct], bins=50, alpha=0.7, label='Correct', color='green')
ax.hist(max_probs[~correct], bins=50, alpha=0.7, label='Incorrect', color='red')
ax.set_xlabel('Max Probability')
ax.set_ylabel('Count')
ax.set_title('Confidence Distribution: Correct vs Incorrect Predictions')
ax.legend()
ax.axvline(x=0.85, color='black', linestyle='--', label='Suggested threshold')
plt.tight_layout()
plt.show()

## 11. Per-Class Accuracy Breakdown

In [ ]:
print(f"{'Agent':<25s} {'Accuracy':<10s} {'Avg Conf':<10s} {'Samples':<10s}")
print('-' * 55)
for i, name in enumerate(le.classes_):
    mask = all_labels == i
    acc = (all_preds[mask] == i).mean()
    avg_conf = all_probs[mask, i].mean()
    print(f'{name:<25s} {acc:<10.1%} {avg_conf:<10.3f} {mask.sum():<10d}')

# Show misclassified examples
test_queries = df['query'].iloc[test_idx].tolist()
misclassified = np.where(all_preds != all_labels)[0]

print(f'\n--- Sample Misclassifications ({len(misclassified)} total) ---')
for idx in misclassified[:15]:
    q = test_queries[idx]
    true_label = le.classes_[all_labels[idx]]
    pred_label = le.classes_[all_preds[idx]]
    conf = max_probs[idx]
    print(f'  [{conf:.2f}] "{q[:60]}..." | true={true_label} pred={pred_label}')

## 12. Save Model for Production

In [ ]:
SAVE_DIR = 'agent_classifier_export'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save model weights
torch.save(model.state_dict(), f'{SAVE_DIR}/model.pt')

# Save tokenizer
tokenizer.save_pretrained(SAVE_DIR)

# Save config
config = {
    'model_name': MODEL_NAME,
    'num_classes': NUM_CLASSES,
    'aux_dim': AUX_DIM,
    'max_len': MAX_LEN,
    'dropout': 0.3,
    'label_map': {str(i): name for i, name in enumerate(le.classes_)},
    'class_names': le.classes_.tolist(),
    'confidence_thresholds': {
        'high': 0.85,
        'medium': 0.50,
        'low': 0.0
    },
    'best_val_accuracy': best_val_acc,
    'test_accuracy': (all_preds == all_labels).mean(),
}

with open(f'{SAVE_DIR}/config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Save label encoder
np.save(f'{SAVE_DIR}/label_classes.npy', le.classes_)

print(f'Saved to {SAVE_DIR}/')
for fname in os.listdir(SAVE_DIR):
    size = os.path.getsize(f'{SAVE_DIR}/{fname}')
    print(f'  {fname}: {size/1024:.1f} KB')

In [ ]:
# Zip and download
!zip -r agent_classifier_export.zip agent_classifier_export/

from google.colab import files
files.download('agent_classifier_export.zip')
print('Download started — save this zip to your project.')

## 13. Inference Demo
Test the model with custom queries.

In [ ]:
def predict(query, top_k=3):
    """Predict which agent should handle this query."""
    model.eval()

    # Tokenize
    enc = tokenizer(query, max_length=MAX_LEN, padding='max_length',
                    truncation=True, return_tensors='pt')
    ids = enc['input_ids'].to(device)
    mask = enc['attention_mask'].to(device)

    # Aux features
    aux = torch.tensor([extract_aux_features(query)], dtype=torch.float32).to(device)

    # Predict
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            logits = model(ids, mask, aux)
        probs = torch.softmax(logits.float(), dim=1)[0]

    # Top-k results
    top_probs, top_indices = probs.topk(top_k)

    conf = top_probs[0].item()
    if conf >= 0.85:
        action = 'ROUTE DIRECTLY'
    elif conf >= 0.50:
        action = 'ROUTE (monitor)'
    else:
        action = 'ASK CLARIFICATION'

    # Check for multi-agent (top-2 are close)
    if top_k >= 2 and (top_probs[0] - top_probs[1]).item() < 0.15:
        action = 'MULTI-AGENT (ambiguous)'

    print(f'Query: "{query}"')
    print(f'Action: {action}')
    for i in range(top_k):
        name = le.classes_[top_indices[i]]
        p = top_probs[i].item()
        bar = '#' * int(p * 30)
        print(f'  {name:<25s} {p:>6.1%} {bar}')
    print()

In [ ]:
# Test with diverse queries
test_queries = [
    # Clear-cut cases
    "show me fleet summary",
    "Rajesh Kumar ka performance kaisa hai",
    "show trip 45892 details",
    "Raipur to Nagpur route ka performance",
    "vehicle CG04MC9150 ki details",
    "predict ETA from Mumbai to Pune",
    "scan last 7 days for anomalies",
    "demand forecast for Delhi to Jaipur route",
    "best route from Hyderabad to Bangalore",
    "is driver 4521 safe to dispatch",

    # Edge cases
    "how many trips did driver 321 do",  # driver_performance (not trip_management)
    "list all trips on Mumbai-Pune route",  # trip_management (not route_intelligence)
    "fleet fatigue summary",  # driver_safety (not fleet_overview)
    "which route will be busiest next week",  # demand_forecasting (not route_intelligence)
    "recommend driver for Kolkata to Bhubaneswar",  # route_optimization

    # Hinglish
    "bhai fleet ka kya haal hai",
    "driver ko rest chahiye kya check karo",

    # Typos
    "shwo me flee sumary",
    "eta predction mumbai pune",
]

for q in test_queries:
    predict(q)